<a href="https://colab.research.google.com/github/adityaarya2004/AI-Code-to-PseudoCode-converter/blob/main/full_working_code%20for%20coingeko%20api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!pip install gspread requests --quiet

from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
import requests
import time

# ==========================
# CONNECT GOOGLE SHEET
# ==========================

creds, _ = default()
gc = gspread.authorize(creds)

SHEET_NAME = "Crypto Tracker"   # 🔁 change if needed
sheet = gc.open(SHEET_NAME).sheet1

# ==========================
# READ TOKENS
# ==========================

tokens = sheet.col_values(1)[1:]
tokens = [t for t in tokens if t]

symbols_needed = set([t.split("/")[0].upper() for t in tokens])

print("Total tokens in sheet:", len(symbols_needed))

# ==========================
# FETCH MARKET DATA
# ==========================

symbol_data = {}

page = 1
max_pages = 60   # 60 × 250 = 15,000 coins

while page <= max_pages and len(symbol_data) < len(symbols_needed):

    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": 250,
        "page": page
    }

    r = requests.get(url, params=params)

    # Handle rate limit
    if r.status_code == 429:
        print("Rate limited. Sleeping 5 sec...")
        time.sleep(5)
        continue

    if r.status_code != 200:
        print("Error on page", page, "- retrying...")
        time.sleep(2)
        continue

    data = r.json()

    if not data:
        break

    for coin in data:
        sym = coin["symbol"].upper()

        if sym in symbols_needed:

            # If duplicate symbol → keep highest market cap
            if (
                sym not in symbol_data
                or (coin["market_cap"] or 0) > symbol_data[sym]["market_cap"]
            ):
                symbol_data[sym] = {
                    "price": coin["current_price"],
                    "change": coin["price_change_percentage_24h"],
                    "volume": coin["total_volume"],
                    "market_cap": coin["market_cap"]
                }

    print(f"Scanned page {page} | Found {len(symbol_data)} / {len(symbols_needed)}")

    page += 1
    time.sleep(1.2)

# ==========================
# PREPARE OUTPUT (E–H)
# ==========================

output = []

for token in tokens:

    base = token.split("/")[0].upper()
    data = symbol_data.get(base)

    if data:
        output.append([
            data["price"],
            data["change"],
            data["volume"],
            data["market_cap"]
        ])
    else:
        output.append([None, None, None, None])

# ==========================
# UPDATE SHEET
# ==========================

sheet.update("E2:H" + str(len(output) + 1), output)

print("✅ Update Complete")

Total tokens in sheet: 609
Scanned page 1 | Found 153 / 609
Scanned page 2 | Found 258 / 609
Scanned page 3 | Found 350 / 609
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Scanned page 4 | Found 418 / 609
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Rate limited. Sleeping 5 sec...
Scanned page 5 | Found 467 / 609
Scanned page 6 | Found 506 / 609
Scanned

/tmp/ipython-input-145/3370008545.py:116: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update("E2:H" + str(len(output) + 1), output)


✅ Update Complete
